## Cross encoder - Doug blog posts

*To use this notebook, go to Kernel -> Change Kernel. Select switch to **Agentic Search Course Environment** *

This uses a MSMarco cross encoder to rerank Doug's blog posts.

A cross encoder is a common reranking method using a transformer model like BERT. It pairs the question with the answer, and asks the cross-encoder "Is this relevant?"

Often we use a pre-trained cross-encoder on an existing dataset. Here we use MSMarco passage rertrieval cross encoder. We see how well it works on Doug's blog posts and those from [OpenSource Connections](http://opensourceconnections.com)

In [1]:
from cheat_at_search.doug_blog_data import corpus

In [2]:
corpus

,doc_id,title,description,publication_date
0,0,"Write for yourself, not the audience","Write *to* an audience, don't write *for* them...",2019-08-26
1,1,Ian and Daddy at One World Trade Center & 9/11...,Ian and Daddy on their first day in New York C...,2020-02-29
2,2,Ian and Daddy Visit Lady Liberty,"On February 29th, Ian and Daddy took our first...",2020-02-29
3,3,Kill Your Twitter Addiction With This One Weir...,I have a confession. I sometimes get wrapped u...,2020-04-05
4,4,Murray running with Daddy with pretty gardens!,"On April 22nd, Murray went running with Daddy....",2020-04-22
...,...,...,...,...
952,952,"E-commerce Search for Product Managers (MICES,...",We’re partnering with TUDOCK GmbH to deliver ...,2019-06-18
953,953,Search Relevance Organizational Maturity Model...,"OSC are sponsoring MICES 2019, the Mix-camp f...",2019-06-19
954,954,Elasticsearch 'Think Like a Relevance Engineer...,"Over two days, gives you a solid foundation f...",2019-08-06
955,955,Solr 'Think Like a Relevance Engineer' Training!,"Over two days, gives you a solid foundation f...",2019-08-06


## Retrieve initial candidates w/ BM25

* Retrieve candidates from MSMarco
* Then we'll rerank them with SentenceTransformer's cross encoder

We notice that BM25 doesn't quite capture the idea of "multi field search"

In [3]:
from searcharray import SearchArray
from cheat_at_search.strategy import SearchStrategy
from cheat_at_search.tokenizers import snowball_tokenizer
import numpy as np


class BM25Strategy(SearchStrategy):

    def __init__(self, corpus, workers=1):
        if 'body_snowball' not in corpus.columns:
            corpus['body_snowball'] = SearchArray.index(corpus['description'],
                                                        tokenizer=snowball_tokenizer)
        if 'title_snowball' not in corpus.columns:
            corpus['title_snowball'] = SearchArray.index(corpus['title'],
                                                         tokenizer=snowball_tokenizer)

        super().__init__(corpus, workers)

    def search(self, query, k=10):
        """Dumb baseline lexical search"""
        tokenized = snowball_tokenizer(query)
        bm25_scores = np.zeros(len(self.corpus))
        for token in tokenized:
            bm25_scores += self.corpus['body_snowball'].array.score(
                token)
            bm25_scores += self.corpus['title_snowball'].array.score(
                token)
        top_k = np.argsort(-bm25_scores)[:k]
        scores = bm25_scores[top_k]
        return top_k, scores

bm25_strategy = BM25Strategy(corpus)


2026-03-19 16:29:50,098 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-19 16:29:50,100 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-19 16:29:50,100 - searcharray.indexing - INFO - Tokenizing 957 documents
2026-03-19 16:29:50,473 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-19 16:29:50,487 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-19 16:29:50,491 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-19 16:29:50,530 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-19 16:29:50,557 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-19 16:29:50,558 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-19 16:29:50,573 - searcharray.indexing - INFO - Indexing from tokenization complete
2026-03-19 16:29:50,603 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-19 16:29:50,608 - searcharray.indexing - INFO - 0 Batch Star

In [4]:
query = "BM25 multi field"
top_k, scores = bm25_strategy.search(query)

corpus.iloc[top_k]

,doc_id,title,description,publication_date,body_snowball,title_snowball
121,121,Can BM25 be a probability?,We've built on a BM25 foundation. These unboun...,2026-03-06,"Terms({'11', 'each', 'forc', 'worth', 'of', 'm...","Terms({'bm25', 'be', 'a', 'can', 'probabl'})"
768,768,BM25 The Next Generation of Lucene Relevance,There’s something new cooking in how Lucene s...,2015-10-16,"Terms({'lie', 'approxim', 'scari', 'of', 'cert...","Terms({'lucen', 'bm25', 'the', 'next', 'genera..."
111,111,BM25F from scratch,Expanding on my [Cheat at Search Essentials](h...,2025-09-18,"Terms({'each', 'vs', 'frequent', 'fourth', 'of...","Terms({'bm25f', 'from', 'scratch'})"
125,125,Remove term frequency from title fields,Users want to know if a document is ***about**...,2025-12-15,"Terms({'expert', 'the', 'infer', '2015', 'gene...","Terms({'titl', 'field', 'frequenc', 'from', 'r..."
835,835,What's up with multi-term synonyms in Solr?,There were some questions floating around the...,2016-06-23,"Terms({'with', 'float', 'be', 'the', 'anoth', ...","Terms({'with', 'up', 'in', 's', 'synonym', 'so..."
91,91,Elasticsearch Hybrid Search Recipes - Benchmarked,Previously I wrote a blog article on [Elastics...,2025-03-13,"Terms({'11', 'each', 'sandbox', 'of', 'blog', ...","Terms({'search', 'elasticsearch', 'hybrid', 'b..."
718,718,Elasticsearch Cross Field Search Is A Lie,"\n At OpenSource Connections, We Do What We...",2015-03-19,"Terms({'each', 'lie', 'approxim', 'connect', '...","Terms({'search', 'field', 'a', 'elasticsearch'..."
731,731,Deep Dive into Elasticsearch Cross Field Search,\n \n Crossing streams -- problem. Cross...,2015-05-27,"Terms({'turnbull""', 'each', 'case', 'of', 'blo...","Terms({'search', 'field', 'elasticsearch', 'cr..."
160,160,Bayesian BM25 is cool,Look at this math and grasp at its majesty:\n\...,2026-03-16,"Terms({'grasp', 'be', '11', 'the', 'researchg'...","Terms({'cool', 'bm25', 'is', 'bayesian'})"
156,156,High IDF doesn't always mean relevant search,Rare terms have high inverse document frequenc...,2026-03-04,"Terms({'with', 'be', 'just', 'the', 'shove', '...","Terms({'search', 'mean', 't', 'doesn', 'alway'..."


## Now use a cross encoder

With a small cross encoder, trained on msmarco, we see improvements.

However you'll notice one problem: **speed** - you'll notice the latency, given you're likely running this notebook on a CPU. These models work better run on a GPU. Reranking 100 candidates with even a small model (MiniLM) takes noticable time!

In [ ]:
from sentence_transformers import CrossEncoder

# Load a pretrained cross-encoder model
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

class CrossEncoderStrategy(SearchStrategy):

    def __init__(self, corpus, num_candidates=100,
                 workers=1):
        if 'body_snowball' not in corpus.columns:
            corpus['body_snowball'] = SearchArray.index(corpus['description'],
                                                        tokenizer=snowball_tokenizer)
        if 'title_snowball' not in corpus.columns:
            corpus['title_snowball'] = SearchArray.index(corpus['title'],
                                                         tokenizer=snowball_tokenizer)
        self.num_candidates = num_candidates
        super().__init__(corpus, workers)

    def search(self, query, k=10):
        """Dumb baseline lexical search"""
        tokenized = snowball_tokenizer(query)
        bm25_scores = np.zeros(len(self.corpus))
        for token in tokenized:
            bm25_scores += self.corpus['body_snowball'].array.score(
                token)
            bm25_scores += self.corpus['title_snowball'].array.score(
                token)
        candidates = np.argsort(-bm25_scores)[:self.num_candidates]
        scores = bm25_scores[candidates]

        # Rerank with cross econder
        pairs = [(query, self.corpus.iloc[i]['description']) for i in candidates]
        scores = model.predict(pairs)
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        top_k = [i for i, _ in ranked[:k]]
        scores = [s for _, s in ranked[:k]]
        return top_k, scores

cross_encoder_strategy = CrossEncoderStrategy(corpus, workers=4)


In [6]:
query = "BM25 multi field"
top_k, scores = cross_encoder_strategy.search(query)

corpus.iloc[top_k]

,doc_id,title,description,publication_date,body_snowball,title_snowball
111,111,BM25F from scratch,Expanding on my [Cheat at Search Essentials](h...,2025-09-18,"Terms({'each', 'vs', 'frequent', 'fourth', 'of...","Terms({'bm25f', 'from', 'scratch'})"
856,856,BM25F in Lucene with BlendedTermQuery,As part of the London hack days Diego Ceccare...,2016-10-19,"Terms({'each', 'approxim', 'of', 'simpli', 's'...","Terms({'lucen', 'with', 'in', 'bm25f', 'blende..."
156,156,High IDF doesn't always mean relevant search,Rare terms have high inverse document frequenc...,2026-03-04,"Terms({'with', 'be', 'just', 'the', 'shove', '...","Terms({'search', 'mean', 't', 'doesn', 'alway'..."
158,158,Ugly hack to force BM25 to 0-1,Its convenient to have a lexical score normali...,2026-03-09,"Terms({'with', 'be', 'just', 'the', 'forc', 'm...","Terms({'0', 'ugli', 'bm25', 'forc', '1', 'to',..."
132,132,Field length (norms) matters for relevance,"In a lucene based search engine, BM25 rewards ...",2026-01-07,"Terms({'with', 'be', 'just', 'the', 'case', 'o...","Terms({'length', 'matter', 'field', 'norm', 'r..."
155,155,BM25: probabilistic but not a probability,BM25 models the ***odds*** a term would be obs...,2026-03-02,"Terms({'with', 'be', 'just', 'the', 'each', 'm...","Terms({'but', 'bm25', 'a', 'probabl', 'not', '..."
121,121,Can BM25 be a probability?,We've built on a BM25 foundation. These unboun...,2026-03-06,"Terms({'11', 'each', 'forc', 'worth', 'of', 'm...","Terms({'bm25', 'be', 'a', 'can', 'probabl'})"
768,768,BM25 The Next Generation of Lucene Relevance,There’s something new cooking in how Lucene s...,2015-10-16,"Terms({'lie', 'approxim', 'scari', 'of', 'cert...","Terms({'lucen', 'bm25', 'the', 'next', 'genera..."
113,113,Reasoning boosts search relevance 15-30%,In [my previous article](https://softwaredoug....,2025-10-06,"Terms({'pars', 'each', 'forc', 'case', 'of', '...","Terms({'search', 'reason', '15', 'boost', '30'..."
160,160,Bayesian BM25 is cool,Look at this math and grasp at its majesty:\n\...,2026-03-16,"Terms({'grasp', 'be', '11', 'the', 'researchg'...","Terms({'cool', 'bm25', 'is', 'bayesian'})"
